# Supplementary tables

Assembles the supplementary table sheets from the canonical datasets and the analysis
outputs. Each is written to `chapters/06-supplementary-tables/sheets/` as a CSV;
`build_workbook.py` collects them into one workbook.

| Table | Built here | Source |
| --- | --- | --- |
| ST2 discordant lead variants | yes | Results 4 |
| ST4 gPS against gene sets | yes | Results 5 |
| ST5 ChEMBL target-indication pairs | yes | Results 6 |
| ST6 drug target enrichment | yes | Results 6 |
| ST7 PAV gene-disease pairs with 2-5 areas | yes | Results 6 |
| ST9 therapeutic area assignment | yes | the hierarchy itself |
| ST14 gene-disease associations with gPS | yes | data preparation |
| ST15 cluster membership | yes | data preparation |
| ST16 disease distribution across areas | yes | data preparation |
| ST1 studies, ST10 fine-mapping, ST11 colocalisation | no | need the release study and colocalisation datasets; `02_tables_from_release.ipynb` |
| ST3 GSEA, ST12 L2G performance, ST13 effector genes | no | blocked on missing inputs, see GAPS.md |
| ST8 subgroup analysis | no | needs the therapeutic-area and target-class breakdown of Results 6 |

In [1]:
import pandas as pd
import pyarrow.dataset as ds

from manuscript_methods import clusters, paper

SHEETS = paper.ROOT / "chapters" / "06-supplementary-tables" / "sheets"
SHEETS.mkdir(parents=True, exist_ok=True)


def write(table: pd.DataFrame, name: str) -> None:
    """Write one sheet and report its shape."""
    path = SHEETS / f"{name}.csv"
    table.to_csv(path, index=False)
    print(f"{name}: {table.shape[0]} rows x {table.shape[1]} columns -> {path.name}")


names = clusters.disease_names()
areas = clusters.therapeutic_area_lookup()

## ST9 — therapeutic area assignment

In [2]:
st9 = pd.DataFrame(
    [{"EFO ID": root, "Therapeutic Area": label} for root, label in paper.THERAPEUTIC_AREAS.items()]
    + [{"EFO ID": "N/A", "Therapeutic Area": "other"}]
)
write(st9, "ST9_therapeutic_area_assignment")
st9

ST9_therapeutic_area_assignment: 24 rows x 2 columns -> ST9_therapeutic_area_assignment.csv


,EFO ID,Therapeutic Area
0,EFO_0001444,measurement
1,MONDO_0045024,cancer or benign tumor
2,OTAR_0000018,"genetic, familial or congenital disease"
3,EFO_0005741,infectious disease
4,OTAR_0000009,"injury, poisoning or other complication"
5,OTAR_0000014,pregnancy or perinatal disease
6,MONDO_0024458,disorder of visual system
7,EFO_0000319,cardiovascular disease
8,EFO_0009605,pancreas disease
9,EFO_0010282,gastrointestinal disease


## ST2 — lead variants with discordant pleiotropic effects

In [3]:
features = pd.read_parquet(paper.derived("variant_features"))
discordant = features[(features["uniqueDiseases"] >= 10) & (features["betaSignConcordance"] <= 0.8)].sort_values(
    "uniqueDiseases", ascending=False
)

st2 = pd.DataFrame(
    {
        "variantId": discordant["variantId"],
        "betaSignConcordance": discordant["betaSignConcordance"],
        "uniqueDiseases": discordant["uniqueDiseases"],
        "uniqueTherapeuticAreas": discordant["uniqueTherapeuticAreas"],
        "uniqueDiseaseNames": discordant["diseaseIds"].map(
            lambda ids: "; ".join(sorted({names.get(d, d) for d in (ids if ids is not None else [])}))
        ),
        "uniqueTherapeuticAreaNames": discordant["therapeuticAreas"].map(
            lambda tas: "; ".join(
                sorted({paper.THERAPEUTIC_AREAS.get(t, "other") for t in (tas if tas is not None else [])})
            )
        ),
        "prioritisedGenes": discordant["prioritisedGenes"].map(
            lambda genes: "; ".join(sorted(genes if genes is not None else []))
        ),
    }
)
write(st2, "ST2_discordant_variants")
st2.head()

ST2_discordant_variants: 59 rows x 7 columns -> ST2_discordant_variants.csv


,variantId,betaSignConcordance,uniqueDiseases,uniqueTherapeuticAreas,uniqueDiseaseNames,uniqueTherapeuticAreaNames,prioritisedGenes
11824,19_44908684_T_C,0.659459,85,15,Abdominal Aortic Aneurysm; Acute bronchitis; A...,cardiovascular disease; disorder of visual sys...,ENSG00000130203
22854,2_27508073_T_C,0.627660,34,13,Abnormality of the liver; Addictive alcohol us...,cancer or benign tumor; disorder of visual sys...,ENSG00000084734
22467,22_28725099_A_G,0.694444,32,4,Menorrhagia; Oligomenorrhea; Ovarian cyst; Pol...,cancer or benign tumor; gastrointestinal disea...,ENSG00000183765
17014,1_169549811_C_T,0.800000,30,9,Abnormal thrombosis; Abnormality of limbs; Iro...,cancer or benign tumor; cardiovascular disease...,ENSG00000198734
38507,4_99318162_T_C,0.680851,29,10,Abnormal blood ion concentration; Addictive al...,cancer or benign tumor; cardiovascular disease...,ENSG00000196616


## ST4 — gPS against membership in 21 gene sets

In [4]:
st4 = pd.read_csv(paper.derived("gene_pleiotropy_by_category.csv"))
write(st4, "ST4_gPS_gene_categories")
st4.head()

ST4_gPS_gene_categories: 21 rows x 10 columns -> ST4_gPS_gene_categories.csv


,category,label,odds_ratio,log_odds_ratio,ci_lower,ci_upper,log_ci_lower,log_ci_upper,p_value,fdr
0,Drosophila distant orthologs,Drosophila distant orthologs (830/41.9%),0.800112,-0.223004,0.732881,0.873510,-0.310773,-0.135236,6.361073e-07,1.113188e-06
1,Q1 LoF constraint,Q1 LoF constraint (4526/33.2%),0.854752,-0.156944,0.818083,0.893065,-0.200791,-0.113096,2.294544e-12,6.023177e-12
2,Essential Gene (DepMap),Essential Gene (DepMap) (1489/35.3%),0.860247,-0.150535,0.802621,0.922011,-0.219873,-0.081198,2.088882e-05,3.374348e-05
3,Non-essential Gene (DepMap),Non-essential Gene (DepMap) (766/31.5%),0.860816,-0.149874,0.778416,0.951939,-0.250494,-0.049254,3.507303e-03,5.260954e-03
4,Cellular lethal (FUSIL),Cellular lethal (FUSIL) (415/39.0%),0.875911,-0.132491,0.776079,0.988585,-0.253500,-0.011481,3.187947e-02,4.184180e-02


## ST5 — all ChEMBL target-indication pairs with genetic support

In [5]:
st5 = pd.read_csv(paper.derived("df_for_enrichment_regression.csv"))
write(st5, "ST5_chembl_ti_pairs")
print(
    "approved pairs:",
    int(st5["outcome"].sum()),
    "| approved with genetic support:",
    int(((st5["outcome"] == 1) & (st5["geneticSupport"] == 1)).sum()),
)
st5.head()

ST5_chembl_ti_pairs: 37377 rows x 11 columns -> ST5_chembl_ti_pairs.csv
approved pairs: 4564 | approved with genetic support: 242


,targetId,diseaseId,indirect_assoc_score,max_beta,min_maf,max_vep,maxClinicalPhase,uniqueDiseases,uniqueTherapeuticAreas,outcome,geneticSupport
0,ENSG00000007314,EFO_0000555,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
1,ENSG00000007314,EFO_0004699,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0
2,ENSG00000007314,EFO_0801084,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
3,ENSG00000010310,EFO_0003884,0.0,0.0,0.0,0.0,2.0,15.0,9.0,0,0
4,ENSG00000012504,MONDO_0019052,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1,0


## ST6 — drug target enrichment results

In [6]:
forest = pd.read_csv(paper.derived("drug_enrichment_subsets_vs_full_l2g.csv"))
resources = pd.read_csv(paper.derived("drug_enrichment_other_resources.csv"))
st6 = pd.concat([forest, resources], ignore_index=True)
write(st6, "ST6_drug_target_enrichment")
st6[["datasource", "clinicalPhase", "odds_ratio", "Relative success", "yes_evid-high_clinphase"]].round(3)

ST6_drug_target_enrichment: 29 rows x 19 columns -> ST6_drug_target_enrichment.csv


,datasource,clinicalPhase,odds_ratio,Relative success,yes_evid-high_clinphase
0,full_l2g,4+,3.619,2.765,242
1,PAV_subEvid,4+,6.048,3.791,72
2,PAV_base,4+,3.092,2.480,170
3,BigEffect_subEvid,4+,4.628,3.241,39
4,BigEffect_base,4+,3.473,2.689,203
5,rare_subEvid,4+,6.994,4.097,29
6,rare_base,4+,3.395,2.647,213
7,low-gPS-5_subEvid,4+,4.798,3.314,86
8,high-gPS_subEvid,4+,2.968,2.409,104
9,TA-1_subEvid,4+,4.291,3.064,22


## ST7 — gene-disease associations supported by a PAV with 2 to 5 therapeutic areas

One row per credible set supporting such an association, as published.

In [7]:
gene_table = pd.read_parquet(
    paper.derived("gene_table"), columns=["geneId", "approvedSymbol", "uniqueTherapeuticAreas"]
)
window = gene_table[gene_table["uniqueTherapeuticAreas"].between(2, 5)]

l2g = pd.read_parquet(
    paper.derived("prioritised_genes_diseases"),
    columns=[
        "geneId",
        "studyLocusId",
        "studyId",
        "score",
        "eQTL_coloc",
        "pQTL_coloc",
        "VEP",
        "distanceTSS",
        "variantId",
        "maf",
        "absBeta",
        "diseaseIds",
        "year",
    ],
)
st7 = l2g[(l2g["VEP"] == 1) & l2g["geneId"].isin(set(window["geneId"]))].merge(window, on="geneId", how="left")
write(st7, "ST7_pav_gene_disease_pairs")

pairs = st7[["geneId", "diseaseIds"]].explode("diseaseIds").dropna().drop_duplicates()
print("distinct gene-disease associations:", len(pairs), "(manuscript: 2734)")
st7.head(3)

ST7_pav_gene_disease_pairs: 4316 rows x 15 columns -> ST7_pav_gene_disease_pairs.csv


distinct gene-disease associations: 2734 (manuscript: 2734)


,geneId,studyLocusId,studyId,score,eQTL_coloc,pQTL_coloc,VEP,distanceTSS,variantId,maf,absBeta,diseaseIds,year,approvedSymbol,uniqueTherapeuticAreas
0,ENSG00000184216,2db27a81275534ce51bb399852b3933c,FINNGEN_R12_AUTOIMMUNE,0.424932,1,0,1,1,X_154018741_A_G,0.152254,0.039383,[EFO_0005140],2024,IRAK1,3
1,ENSG00000133466,0c79a22b919ee762992d4297c7afade7,FINNGEN_R12_AUTOIMMUNE,0.899293,1,0,1,1,22_37185445_C_A,0.390084,0.039157,[EFO_0005140],2024,C1QTNF6,5
2,ENSG00000144802,d5924c489e8eb817d9dc364c7ac2ce6c,FINNGEN_R12_AUTOIMMUNE,0.906680,0,0,1,1,3_101852100_G_C,0.014385,0.114544,[EFO_0005140],2024,NFKBIZ,4


## ST14 — every gene-disease association with gPS and area count

In [8]:
gene_table = pd.read_parquet(paper.derived("gene_table"))
associations = (
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["geneId", "diseaseIds"])
    .explode("diseaseIds")
    .dropna()
    .drop_duplicates()
    .rename(columns={"diseaseIds": "diseaseId"})
)
st14 = associations.merge(
    gene_table[["geneId", "approvedSymbol", "uniqueDiseases", "uniqueTherapeuticAreas"]], on="geneId", how="left"
)
st14["diseaseName"] = st14["diseaseId"].map(names)
st14["therapeuticArea"] = st14["diseaseId"].map(lambda d: paper.THERAPEUTIC_AREAS.get(areas.get(d, "other"), "other"))
st14 = st14.rename(columns={"uniqueDiseases": "gPS", "uniqueTherapeuticAreas": "numberOfTherapeuticAreas"})
write(st14, "ST14_gene_disease_with_gps")
st14.head()

ST14_gene_disease_with_gps: 36858 rows x 7 columns -> ST14_gene_disease_with_gps.csv


,geneId,diseaseId,approvedSymbol,gPS,numberOfTherapeuticAreas,diseaseName,therapeuticArea
0,ENSG00000174125,EFO_0008510,TLR1,19,7,Lyme disease,infectious disease
1,ENSG00000124935,EFO_0008510,SCGB1D2,2,1,Lyme disease,infectious disease
2,ENSG00000132693,EFO_0000771,CRP,9,3,bacterial disease,infectious disease
3,ENSG00000138031,EFO_0000771,ADCY3,10,6,bacterial disease,infectious disease
4,ENSG00000130203,EFO_0000771,APOE,107,16,bacterial disease,infectious disease


## ST15 — diseases linked through each colocalisation cluster

In [9]:
st15 = pd.read_parquet(paper.derived("cluster_membership"))
write(st15, "ST15_cluster_membership")
print("clusters:", st15["cluster_id"].nunique())
st15.head()

ST15_cluster_membership: 42918 rows x 6 columns -> ST15_cluster_membership.csv
clusters: 20041


,cluster_id,leadVariants,diseaseId,diseaseName,therapeuticArea,vPS
0,0,16_89579029_G_T,EFO_0000756,melanoma,cancer or benign tumor,2
1,0,16_89579029_G_T,EFO_0004279,suntan,other,2
2,1,2_66523432_G_T,EFO_0004270,restless legs syndrome,nervous system disease,5
3,1,2_66523432_G_T,EFO_0004280,movement disorder,nervous system disease,5
4,1,2_66523432_G_T,EFO_0004698,insomnia,nervous system disease,5


## ST16 — distribution of diseases across therapeutic areas

Two universes: every disease term carried by a qualifying study, and the gPS disease list,
which is every disease term carrying at least one credible set with an L2G-prioritised gene.

In [10]:
qualifying_terms = set(
    ds.dataset(paper.derived("qualifying_gwas_studies"), format="parquet")
    .to_table(columns=["diseaseIds"])
    .to_pandas()["diseaseIds"]
    .explode()
    .dropna()
)
gps_terms = set(
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["diseaseIds"])["diseaseIds"]
    .explode()
    .dropna()
)
print("qualifying disease terms:", len(qualifying_terms), "| gPS disease terms:", len(gps_terms))


def counts(terms):
    """Disease terms per therapeutic area, measurements excluded."""
    labels = pd.Series([areas.get(t, "other") for t in terms])
    return labels[labels != paper.MEASUREMENT].value_counts()


qualifying_counts, gps_counts = counts(qualifying_terms), counts(gps_terms)
st16 = pd.DataFrame(
    {
        "Root ID": list(paper.THERAPEUTIC_AREAS) + ["other"],
        "Therapeutic Area": list(paper.THERAPEUTIC_AREAS.values()) + ["other (no area root)"],
    }
)
st16 = st16[st16["Root ID"] != paper.MEASUREMENT]
st16["Diseases (qualifying dataset)"] = st16["Root ID"].map(qualifying_counts).fillna(0).astype(int)
st16["Diseases (gPS list)"] = st16["Root ID"].map(gps_counts).fillna(0).astype(int)
for column in ["qualifying dataset", "gPS list"]:
    total = st16[f"Diseases ({column})"].sum()
    st16[f"% of {column}"] = (100 * st16[f"Diseases ({column})"] / total).round(1)
st16 = st16.sort_values("Diseases (gPS list)", ascending=False)
write(st16, "ST16_ta_distribution")
st16

qualifying disease terms: 2320 | gPS disease terms: 1394
ST16_ta_distribution: 23 rows x 6 columns -> ST16_ta_distribution.csv


,Root ID,Therapeutic Area,Diseases (qualifying dataset),Diseases (gPS list),% of qualifying dataset,% of gPS list
23,other,other (no area root),586,303,25.3,21.7
1,MONDO_0045024,cancer or benign tumor,350,240,15.1,17.2
7,EFO_0000319,cardiovascular disease,161,116,6.9,8.3
19,EFO_0000618,nervous system disease,175,90,7.5,6.5
6,MONDO_0024458,disorder of visual system,117,85,5.0,6.1
17,EFO_0000540,immune system disease,111,75,4.8,5.4
15,OTAR_0000006,musculoskeletal or connective tissue disease,97,66,4.2,4.7
9,EFO_0010282,gastrointestinal disease,95,59,4.1,4.2
3,EFO_0005741,infectious disease,142,53,6.1,3.8
4,OTAR_0000009,"injury, poisoning or other complication",65,39,2.8,2.8
